In [1]:
import os
os.makedirs("going_modular",exist_ok=True)


In [7]:
import os
import zipfile
from pathlib import Path
import requests

data_path = Path("going_modular/data/")
image_path = data_path / "pizza-sushi-steak"

if image_path.is_dir():
    print(f"{image_path} exist.")
else:
    image_path.mkdir(parents=True,exist_ok=True)

with open(data_path/"pizza_steak_sushi.zip","wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
    f.write(request.content)

with zipfile.ZipFile(data_path/"pizza_steak_sushi.zip","r") as zip_ref:
    zip_ref.extractall(image_path)

os.remove(data_path/"pizza_steak_sushi.zip")





going_modular\data\pizza-sushi-steak exist.


In [8]:
train_dir = image_path / "train"
test_dir = image_path / "test"
train_dir,test_dir

(WindowsPath('going_modular/data/pizza-sushi-steak/train'),
 WindowsPath('going_modular/data/pizza-sushi-steak/test'))

In [12]:
from torchvision import datasets,transforms
data_transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor()
])
train_data = datasets.ImageFolder(train_dir,
                                  transform=data_transform)
test_data = datasets.ImageFolder(test_dir,
                                 transform=data_transform)

print(f"Train data:\n{train_data}\nTest data:\n{test_data}")

Train data:
Dataset ImageFolder
    Number of datapoints: 225
    Root location: going_modular\data\pizza-sushi-steak\train
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )
Test data:
Dataset ImageFolder
    Number of datapoints: 75
    Root location: going_modular\data\pizza-sushi-steak\test
    StandardTransform
Transform: Compose(
               Resize(size=(64, 64), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )


In [17]:
class_name = train_data.classes
class_dict = train_data.class_to_idx
class_name,class_dict

(['pizza', 'steak', 'sushi'], {'pizza': 0, 'steak': 1, 'sushi': 2})

In [18]:
len(train_data),len(test_data)

(225, 75)

In [19]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(dataset=train_data,
                              batch_size=32,
                              shuffle=True,
                              num_workers=1)

test_dataloader = DataLoader(dataset=test_data,
                             batch_size=32,
                             num_workers=1,
                             shuffle=False)

train_dataloader,test_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x212f546fe00>,
 <torch.utils.data.dataloader.DataLoader at 0x212934cbed0>)

In [20]:
img,label = next(iter(train_dataloader))
img.shape,label.shape

(torch.Size([32, 3, 64, 64]), torch.Size([32]))

In [22]:
%%writefile going_modular/data_setup.py

import os 
from torch.utils.data import DataLoader
from torchvision import datasets,transforms

def create_dataloaders(
    train_dir:str,
    test_dir:str,
    transform:transforms.Compose,
    batch_size:int,
    num_workers:int
):

train_data = datasets.ImageFolder(train_dir,transform=transform)
teat_data = datasets.ImageFolder(test_dir,transform=transform)

class_name=train_data.classes

train_dataloader = DataLoader(
    train_data,
    batch_size=batch_size
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)
return train_dataloader,test_dataloader,class_name

Overwriting going_modular/data_setup.py


In [ ]:
import torch
from torch import nn

class TinyVGG(nn.Module):
    def __init__(self,input_shape:int,output_shape:int,hidden_shape:int)-> None :
        super().__init__():
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(input_shape,hidden_shape,3,1,0),
            nn.ReLU(),
            nn.Conv2d(hidden_shape,hidden_shape,3,1,0)
            nn.ReLU(),
            nn.MaxPool2d(kernel)
        )

